<a href="https://colab.research.google.com/github/stephleao/wmc-desafio-streaming-churn/blob/main/%5BNina_da_Hora%5D_Probabilidade_e_Amostragem_%7C_Bootcamp_Data_Analytics_2026_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

# 1. CARREGAMENTO DA BASE CORRIGIDO (Adicionado o sep=';')
url = 'https://raw.githubusercontent.com/stephleao/wmc-desafio-streaming-churn/refs/heads/main/clientes.csv'
df = pd.read_csv(url, sep=';')

print("=== 1. DIAGNÓSTICO INICIAL DOS DADOS ===")
print(f"Total de registros originais: {df.shape[0]} linhas e {df.shape[1]} colunas\n")

# 2. VERIFICAÇÃO DE SAÚDE DOS DADOS
print("--- Tipos de Dados ---")
print(df.dtypes)

print("\n--- Valores Ausentes (Nulos) por Coluna ---")
print(df.isna().sum())

print(f"\nLinhas duplicadas identificadas: {df.duplicated().sum()}")

# 3. LIMPEZA (Removendo duplicados, se existirem)
df = df.drop_duplicates()

# 4. CRIAÇÃO DOS GRUPOS COM BASE NO TEMPO DE ASSINATURA DOS CANCELADOS
# Consideramos quem cancelou (cancelou == 1 ou 'Sim') e mapeamos o tempo
# Adaptando a regra do negócio para as colunas reais da sua base:
condicoes = [
    (df['cancelou'].isin([1, 'Sim', 'sim', 'S']) & (df['tempo_assinatura_meses'] <= 6)),
    (df['cancelou'].isin([1, 'Sim', 'sim', 'S']) & (df['tempo_assinatura_meses'] > 24))
]
nomes_grupos = ['Ultimos 6 meses', 'Mais de 24 meses']

# Criando a nova coluna de segmentação
df['grupo_cancelamento'] = np.select(condicoes, nomes_grupos, default='Outros Periodos / Ativos')

print("\n=== 2. QUANTIDADE DE CLIENTES POR GRUPO ===")
print(df['grupo_cancelamento'].value_counts())

In [ ]:
# Ver os nomes reais de todas as colunas da base
print(df.columns.tolist())

In [ ]:
#IMPORTAÇÃO BIBLIOTECAS COMPLENTARES
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

## Análise 1 — Taxa de cancelamento por tempo de assinatura - Comportamental

✔ Foco da análise

Avalia a taxa de cancelamento em diferentes faixas de tempo de assinatura, identificando períodos de maior risco de churn.

✔ O que essa análise responde

* Em qual faixa de tempo ocorre maior churn
* Clientes com pouco tempo de assinatura cancelam mais?
* O risco de cancelamento diminui com o tempo de uso?
* Qual é a taxa de cancelamento em cada grupo de tempo

✔ Pergunta de negócio

O tempo de permanência influencia a probabilidade de cancelamento?

✔ Tipo de insight gerado

* Probabilidade de churn
* Comportamento ao longo do tempo
* Identificação de períodos críticos

In [ ]:
# Cálculo do percentual de clientes que cancelaram e permaneceram ativos.
# Essa análise apresenta a distribuição da variável alvo (cancelamento),
# permitindo avaliar o equilíbrio entre os grupos da base.

percentual = (
    df['cancelou']
    .value_counts(normalize=True)
    .mul(100)
    .rename({
        0: 'Não cancelou',
        1: 'Cancelou'
    })
)

plt.figure(figsize=(6, 4))

ax = sns.barplot(
    x=percentual.index,
    y=percentual.values,
    hue=percentual.index,
    palette='rocket',
    legend=False
)

for i, valor in enumerate(percentual.values):
    ax.text(i, valor, f'{valor:.1f}%', ha='center', va='bottom')

plt.title('Percentual de Cancelamento')
plt.xlabel('Status')
plt.ylabel('Percentual (%)')

plt.ylim(0, percentual.max() + 10)

plt.show()

In [ ]:
# Criação de faixas de tempo de assinatura.
# Os clientes são agrupados conforme o tempo de permanência para avaliar
# como a taxa de cancelamento varia ao longo da jornada.

df['grupo_tempo'] = pd.cut(
    df['tempo_assinatura_meses'],
    bins=[0, 6, 24, df['tempo_assinatura_meses'].max()],
    labels=['Até 6 meses', '6 a 24 meses', 'Acima de 24 meses']
)

In [ ]:
# Tabela de contingência entre tempo de assinatura e status de cancelamento.
# Apresenta o total de clientes por grupo e o percentual de cancelamento
# dentro de cada faixa de tempo.

tabela_pct = (
    pd.crosstab(
        df['grupo_tempo'],
        df['cancelou'],
        normalize='index'
    ) * 100
).round(2)

tabela_pct = tabela_pct.rename(columns={
    0: 'Não cancelou (%)',
    1: 'Cancelou (%)'
})

total_clientes = df.groupby('grupo_tempo', observed=False).size()

tabela = tabela_pct.copy()
tabela.insert(0, 'Total de clientes', total_clientes)

tabela.loc['Total'] = [
    df.shape[0],
    round((df['cancelou'] == 0).mean() * 100, 2),
    round((df['cancelou'] == 1).mean() * 100, 2)
]

tabela

In [ ]:
# Visualização da taxa de cancelamento por faixa de tempo de assinatura.
# O objetivo é identificar em quais períodos ocorre maior concentração de churn.

percentual = df.groupby('grupo_tempo', observed=False)['cancelou'].mean() * 100

percentual = percentual.reindex([
    'Até 6 meses',
    '6 a 24 meses',
    'Acima de 24 meses'
])

plt.figure(figsize=(6, 4))

ax = sns.barplot(
    x=percentual.index,
    y=percentual.values,
    hue=percentual.index,
    palette='rocket',
    legend=False
)

for i, valor in enumerate(percentual.values):
    ax.text(i, valor, f'{valor:.1f}%', ha='center', va='bottom')

plt.title('Percentual de Cancelamento por Tempo de Assinatura')
plt.xlabel('Grupo de tempo')
plt.ylabel('Percentual de cancelamento (%)')

plt.ylim(0, percentual.max() + 10)

plt.show()

In [ ]:
# Separação da base em clientes ativos e cancelados.
# Essa segmentação será utilizada nas análises comparativas entre os grupos.

cancelaram = df[df["cancelou"] == 1].copy()
ativos = df[df["cancelou"] == 0].copy()

# Estatísticas descritivas das variáveis numéricas da base.
# Apresenta uma visão geral da distribuição dos dados, incluindo medidas
# de tendência central, dispersão, quartis e valores mínimos e máximos.

display(Markdown("## Estatísticas descritivas da base"))
display(df.describe().T.round(2))

display(Markdown("---"))

# Estatísticas descritivas dos clientes cancelados.
# Caracteriza o perfil médio dos clientes que encerraram a assinatura,
# permitindo identificar possíveis padrões associados ao cancelamento.

display(Markdown("## Estatísticas descritivas dos clientes cancelados"))
display(cancelaram.describe().T.round(2))

display(Markdown("---"))

# Estatísticas descritivas dos clientes ativos.
# Caracteriza o perfil médio dos clientes que permaneceram na plataforma,
# servindo como base para comparação com o grupo de clientes cancelados.

display(Markdown("## Estatísticas descritivas dos clientes ativos"))
display(ativos.describe().T.round(2))

In [ ]:
comparacao = pd.DataFrame({
    'Cancelaram': cancelaram[['idade', 'tempo_assinatura_meses',
                              'frequencia_uso_mensal', 'mensalidade']].mean(),
    'Não cancelaram': ativos[['idade', 'tempo_assinatura_meses',
                      'frequencia_uso_mensal', 'mensalidade']].mean()
}).round(2)

comparacao

In [ ]:
# Comparação da distribuição da idade entre clientes ativos e cancelados.
# O boxplot permite analisar mediana, dispersão e possíveis valores extremos.

plt.figure(figsize=(8, 5))

sns.boxplot(
    data=df,
    x='cancelou',
    y='idade',
    hue='cancelou',
    palette='rocket',
    legend=False
)

plt.xticks([0, 1], ['Não cancelou', 'Cancelou'])
plt.title('Distribuição da idade por status de cancelamento')
plt.xlabel('Status')
plt.ylabel('Idade')

plt.show()

In [ ]:
# Comparação da distribuição do tempo de assinatura entre clientes ativos e cancelados.
# Permite verificar diferenças na permanência dos clientes antes do cancelamento.

plt.figure(figsize=(8, 5))

sns.boxplot(
    data=df,
    x='cancelou',
    y='tempo_assinatura_meses',
    hue='cancelou',
    palette='rocket',
    legend=False
)

plt.xticks([0, 1], ['Não cancelou', 'Cancelou'])
plt.title('Tempo de assinatura por status de cancelamento')
plt.xlabel('Status')
plt.ylabel('Tempo de assinatura (meses)')

plt.show()

In [ ]:
# Comparação da distribuição da mensalidade entre clientes ativos e cancelados.
# O boxplot permite identificar diferenças na mediana, dispersão e possíveis
# valores extremos, indicando se o valor pago pode estar associado ao cancelamento.

plt.figure(figsize=(8, 5))

sns.boxplot(
    data=df,
    x='cancelou',
    y='mensalidade',
    hue='cancelou',
    palette='rocket',
    legend=False
)

plt.xticks([0, 1], ['Não cancelou', 'Cancelou'])
plt.title('Distribuição da mensalidade por status de cancelamento')
plt.xlabel('Status')
plt.ylabel('Mensalidade (R$)')

plt.show()

### **Insights:**

1.  **Elevada Taxa de Churn Geral (28%):**
    *   **Insight:** Uma taxa de cancelamento de 28% é significativa e indica que quase um terço dos clientes não permanece na plataforma.
2.  **Churn Concentrado nos Primeiros 6 Meses (37.5%):**
    *   **Insight:** Este é o ponto mais crítico identificado. Clientes em seus primeiros seis meses de assinatura são os mais propensos a cancelar. Isso sugere que a experiência inicial do cliente pode não estar atendendo às expectativas ou que há barreiras iniciais de uso.
3.  **Estabilização e Fidelização a Longo Prazo (25.8% para >24 meses):**
    *   **Insight:** A taxa de churn diminui consideravelmente para clientes com mais de 24 meses de assinatura. Isso indica que, uma vez que o cliente ultrapassa um certo período, ele tende a se fidelizar à plataforma.

### **Limitações:**
1. **Variáveis Atuais Insuficientes para Prever Churn Individual:**
    *   **Insight:** As análises de `idade`, `tempo_assinatura_meses`, `frequencia_uso_mensal` e `mensalidade` não mostraram diferenças significativas entre clientes que cancelaram e os que não cancelaram. Isso significa que, isoladamente, essas variáveis não são bons preditores de churn.
    
        

# Análise 2 — Segmentação comportamental dos clientes

✔ Foco da análise

Descreve o perfil dos clientes em cada grupo de comportamento, comparando ativos e cancelados em relação às principais variáveis da base.

✔ O que essa análise responde

Como são os clientes de cada grupo
* Diferenças de idade entre os grupos
* Diferenças de frequência de uso
* Diferenças de mensalidade
* Diferenças de tempo de permanência

✔ Pergunta de negócio

Quem são os clientes que cancelam e como eles se diferenciam dos demais?

✔ Tipo de insight gerado

* Perfil do cliente
* Segmentação de comportamento
* Características associadas ao churn

In [ ]:
# Criação dos grupos de análise baseados em regras de negócio:
# - Cancelados até 6 meses
# - Cancelados entre 7 e 24 meses
# - Cancelados acima de 24 meses
# - Ativos

condicoes = [
    (df['cancelou'] == 1) & (df['tempo_assinatura_meses'] <= 6),
    (df['cancelou'] == 1) & (df['tempo_assinatura_meses'].between(7, 24)),
    (df['cancelou'] == 1) & (df['tempo_assinatura_meses'] > 24)
]

# Os grupos foram definidos combinando o status de cancelamento com o tempo de 
# assinatura, permitindo identificar padrões de comportamento entre clientes 
# ativos e clientes que cancelaram em diferentes estágios da jornada.
opcoes = [
    'Até 6 meses',
    'Entre 7 e 24 meses',
    'Acima de 24 meses'
]

df['grupo_cancelamento'] = np.select(condicoes, opcoes, default='Ativos')

In [ ]:
# Distribuição dos clientes nos grupos de análise,
# permitindo entender o volume de cada segmento.

display(Markdown("## Distribuição dos clientes por grupo"))

grupo_contagem = df['grupo_cancelamento'].value_counts()

display(grupo_contagem)

In [ ]:
grupo_pct = (df['grupo_cancelamento']
             .value_counts(normalize=True)
             .mul(100)
             .round(2))

plt.figure(figsize=(7,5))

ax = sns.barplot(
    x=grupo_pct.index,
    y=grupo_pct.values,
    hue=grupo_pct.index,
    palette='rocket'
)

for i, v in enumerate(grupo_pct.values):
    ax.text(i, v, f'{v:.1f}%', ha='center', va='bottom')

plt.title('Distribuição percentual dos grupos de clientes')
plt.ylabel('% de clientes')
plt.xlabel('Grupo')
plt.xticks(rotation=30)
plt.show()

In [ ]:
# Comparação das médias das variáveis entre os grupos,
# permitindo identificar diferenças de perfil entre clientes.

display(Markdown("## Perfil médio dos grupos"))

perfil = df.groupby('grupo_cancelamento')[[
    'idade',
    'tempo_assinatura_meses',
    'frequencia_uso_mensal',
    'mensalidade'
]].mean().round(2)

display(perfil)

In [ ]:
plt.figure(figsize=(8,5))

sns.boxplot(
    data=df,
    x='grupo_cancelamento',
    y='idade',
    hue='grupo_cancelamento',
    palette='rocket'
)

plt.title('Distribuição da idade por grupo de clientes')
plt.xticks(rotation=20)
plt.show()

In [ ]:
plt.figure(figsize=(8,5))

sns.boxplot(
    data=df,
    x='grupo_cancelamento',
    y='frequencia_uso_mensal',
    hue='grupo_cancelamento',
    palette='rocket'
)

plt.title('Frequência de uso por grupo de clientes')
plt.xticks(rotation=20)
plt.show()

In [ ]:
plt.figure(figsize=(8,5))

sns.boxplot(
    data=df,
    x='grupo_cancelamento',
    y='mensalidade',
    hue='grupo_cancelamento',
    palette='rocket'
)

plt.title('Mensalidade por grupo de clientes')
plt.xticks(rotation=20)
plt.show()

In [ ]:
plt.figure(figsize=(8,5))

sns.boxplot(
    data=df,
    x='grupo_cancelamento',
    y='tempo_assinatura_meses',
    hue='grupo_cancelamento',
    palette='rocket'
)

plt.title('Tempo de assinatura por grupo de clientes')
plt.xticks(rotation=20)
plt.show()

### **Insights:**

1.  **Distribuição dos Grupos de Cancelamento:** A maioria dos clientes é 'Ativos' (72%), seguido por 'Acima de 24 meses' (16%), 'Entre 7 e 24 meses' (7.5%) e 'Até 6 meses' (4.5%). Isso mostra que a retenção é alta, mas uma parcela significativa de cancelamentos ocorre após 2 anos de assinatura.
2.  **Perfil do Cancelador Inicial (Até 6 meses):** Clientes que cancelam nos primeiros 6 meses tendem a ter: 
    *   Idade ligeiramente menor (média de 40.89 anos).
    *   Menor frequência de uso mensal (média de 11.56).
    *   Mensalidade similar aos ativos, mas com maior dispersão.
    *   Obviamente, menor tempo de assinatura (média de 3.22 meses).
3.  **Perfil do Cancelador Intermediário (Entre 7 e 24 meses):** Clientes neste grupo tendem a ter:
    *   Mensalidades notavelmente mais baixas (média de 47.92), sugerindo que o preço pode ser um fator de churn para este grupo.
    *   Frequência de uso mensal mais alta (média de 20.07), o que é contraintuitivo e pode indicar que mesmo clientes engajados podem cancelar se outros fatores (como preço) não estiverem alinhados.
    *   Idade e tempo de assinatura (média de 16.33 meses) esperados para o grupo.
4.  **Perfil do Cancelador de Longo Prazo (Acima de 24 meses):** Este grupo parece mais próximo dos clientes ativos em termos de idade, frequência de uso e mensalidade, mas possuem tempo de assinatura maior (média de 43.38 meses antes do cancelamento).

### **Limitações:**

1.  **Variáveis Disponíveis:** A análise está restrita às variáveis presentes na base de dados. Fatores externos ou outras características do cliente (ex: tipo de plano, histórico de suporte, feedback do cliente) não foram considerados.
2.  **Definição dos Grupos:** Os limites para os grupos de cancelamento (ex: 'Até 6 meses', 'Entre 7 e 24 meses') foram definidos de forma arbitrária e podem não refletir os pontos de corte ideais para o negócio.
